# 05c — Preprocessing sMRI: Volume Kanonik

**Tujuan.** Mengubah setiap citra T1w menjadi **satu volume kanonik** `float32 (1, 96, 112, 96)` di ruang MNI152 2 mm, sebagai input semua model citra. Input 2D nantinya diturunkan dari volume ini. Tidak ada model yang dilatih.

Pipeline per subjek: reorientasi RAS → N4 → skull stripping HD-BET (GPU, venv terpisah) → registrasi **afin 12 DOF** ke MNI152 → resample 2 mm → clip persentil + z-score di dalam mask → crop box tetap + pad → simpan `.npy`.

| CELL | Isi |
|---|---|
| 1 | CONFIG |
| 2 | Cek lingkungan (versi, GPU, HD-BET) |
| 3 | Muat data, hash awal, QC, citra identik |
| 4 | Template MNI152 dan box crop tetap |
| 5 | Cek header semua citra → `smri_header_check.csv` |
| 6 | Fungsi preprocessing |
| 7 | Validasi runner HD-BET vs CLI (subjek test) |
| 7c | Test 1 subjek (waktu, VRAM, RAM, overlay) |
| 8 | Pilot 20 subjek + estimasi waktu |
| 9 | QC otomatis → `qc_report.csv` + galeri |
| 10 | Cek citra identik |
| 11 | Batch penuh (**kode saja**, `RUN_FULL_BATCH = False`) |
| 12 | Analisis QC vs label/situs (**kode saja**, `RUN_QC_ANALYSIS = False`) |
| 13 | `preprocess_config.json` + verifikasi akhir (hash) |

**Output** (`data/`): `vol/`, `mask/`, `xfm/` (matriks afin), `vol_meta/` (waktu & statistik per subjek), `qc_figures/`, `qc_report.csv`, `failed_subjects.csv`, `smri_header_check.csv`, `preprocess_config.json`, `templates/`.

**Environment:** kernel = Python utama (nibabel, nilearn, antspyx, psutil). HD-BET berjalan di venv `.venv-hdbet` lewat subprocess (`--system-site-packages`, memakai torch CUDA dari Python utama), melalui `src/hdbet_runner.py`: model dimuat sekali per chunk dengan 1 proses preprocessing dan 1 proses export. CLI HD-BET memakai 4 + 8 proses dan menghabiskan RAM laptop 14 GB pada pilot pertama.

### CELL 1 — CONFIG

**Tujuan:** semua path dan parameter di satu tempat. **Output:** variabel global.

Catatan "perlu dicek":
- `HDBET_INPUT_AXCODES = ("L","A","S")`: README HD-BET mensyaratkan orientasi sama dengan template MNI152 standar (`fslreorient2std`). File MNI152 FSL disimpan LAS. Template nilearn yang dipakai untuk registrasi berorientasi RAS (urutan sumbu sama, hanya arah kiri-kanan berbeda). Mask HD-BET dikembalikan ke RAS tanpa interpolasi.
- N4 dan registrasi memakai default antspyx 0.6.3 kecuali yang ditulis di bawah. N4 tanpa mask (default `mask=None`).
- `ANTS_RANDOM_SEED`: menambahkan `--random-seed` ke antsRegistration. ITK tetap multi-thread, jadi hasil bisa berbeda sangat kecil antar-run.
- `IDENTICAL_ATOL`: toleransi selisih volume untuk citra identik (satuan z-score). Pilihan metodologis.

In [ ]:
import datetime
import hashlib
import importlib.metadata as md
import json
import os
import platform
import shutil
import subprocess
import sys
import threading
import time
import traceback
import warnings
from pathlib import Path

ANTS_THREADS = os.cpu_count()
os.environ.setdefault("ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS", str(ANTS_THREADS))

import matplotlib
import matplotlib.pyplot as plt
import psutil
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# ---------- Path ----------
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "dataset").exists() and (PROJECT_ROOT.parent / "dataset").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT                                   # basis smri_path
DATA_DIR = PROJECT_ROOT / "data"
CLEANED_PATH = PROJECT_ROOT / "dataset" / "cleaned_dataset" / "cleaned_master_dataset.csv"
SMRI_QC_PATH = PROJECT_ROOT / "dataset" / "cleaned_dataset" / "smri_qc.csv"
FOLDS_PATH = DATA_DIR / "folds.csv"
AUDIT_FLAGS_PATH = DATA_DIR / "audit_flags.csv"
AUDIT_REPORT_PATH = DATA_DIR / "audit_report.md"
PHENO_CLEAN_PATH = DATA_DIR / "pheno_clean.csv"            # opsional (image_integrity_flag dari 05b)

VOL_DIR, MASK_DIR, XFM_DIR = DATA_DIR / "vol", DATA_DIR / "mask", DATA_DIR / "xfm"
META_DIR, FIG_DIR, TEMPLATE_DIR = DATA_DIR / "vol_meta", DATA_DIR / "qc_figures", DATA_DIR / "templates"
WORK_DIR = DATA_DIR / "work"                               # file antara (boleh dihapus)
HEADER_CHECK_PATH = DATA_DIR / "smri_header_check.csv"
QC_REPORT_PATH = DATA_DIR / "qc_report.csv"
FAILED_PATH = DATA_DIR / "failed_subjects.csv"
CONFIG_JSON_PATH = DATA_DIR / "preprocess_config.json"

VENV_DIR = PROJECT_ROOT / ".venv-hdbet"
VENV_PY = VENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
HDBET_EXE = VENV_DIR / ("Scripts/hd-bet.exe" if os.name == "nt" else "bin/hd-bet")
HDBET_RUNNER = PROJECT_ROOT / "src" / "hdbet_runner.py"

# ---------- Seed ----------
SEED = 42
np.random.seed(SEED)

# ---------- Preprocessing ----------
TARGET_AXCODES = ("R", "A", "S")
HDBET_INPUT_AXCODES = ("L", "A", "S")                      # perlu dicek (lihat catatan)
HDBET_DEVICE = "cuda"                                      # tidak ada fallback otomatis ke CPU
HDBET_TTA = True                                           # default HD-BET (False = --disable_tta)
HDBET_CHUNK = 20                                           # subjek per panggilan HD-BET (model dimuat sekali)
HDBET_MODE = "runner"                                      # "runner" (src/hdbet_runner.py) | "cli" (hd-bet.exe)
HDBET_NP_PREPROCESSING = 1                                 # proses nnU-Net (CLI 2.0.1: 4)
HDBET_NP_EXPORT = 1                                        # proses nnU-Net (CLI 2.0.1: 8)
MIN_FREE_RAM_GB = 1.5                                      # RAM tersedia minimum; di bawah ini proses dihentikan rapi
VALIDATION_DICE_MIN = 0.999                                # runner vs CLI pada subjek test
CLIP_NEGATIVE_BEFORE_N4 = True                             # N4 butuh intensitas positif; dicatat per subjek
N4_PARAMS = dict(shrink_factor=4, convergence={"iters": [50, 50, 50, 50], "tol": 1e-07})   # default antspyx 0.6.3
REG_TRANSFORM = "Affine"                                   # 12 DOF (rigid + scaling + shear), bukan non-linear
ANTS_RANDOM_SEED = SEED
TEMPLATE_RES_REG_MM = 1                                    # template untuk estimasi registrasi
TARGET_RES_MM = 2                                          # grid output
MASK_THRESHOLD = 0.5                                       # mask hasil interpolasi linear -> biner
CLIP_PERCENTILES = (0.5, 99.5)                             # di dalam mask
TARGET_SHAPE = (96, 112, 96)
CROP_MARGIN_VOX = 2                                        # margin di sekitar bbox mask template
KEEP_WORK_FILES = True                                     # pilot: simpan file antara untuk QC
SAVE_QC_FIGURE = True

# ---------- Header check ----------
HEADER_READ_DATA = True                                    # baca voxel untuk cek NaN/Inf
ANISOTROPY_MAX = 1.5
VOXEL_MAX_MM = 1.5

# ---------- Test / pilot ----------
TEST_SUB_ID = None                                         # None = SUB_ID terkecil berstatus PASS
PILOT_N = 20
PILOT_REQUIRED = [50279, 50286]                            # citra identik dari audit 05a

# ---------- QC ----------
OUTLIER_Z = 3.0
MIN_N_GLOBAL_Z = 20
MIN_N_SITE_Z = 15
IDENTICAL_ATOL = 1e-3
GALLERY_N = 12

# ---------- Run flags ----------
RUN_FULL_BATCH = False                                     # batch penuh dijalankan manual
RUN_QC_ANALYSIS = False                                    # setelah batch penuh

assert SEED == 42 and REG_TRANSFORM == "Affine" and HDBET_MODE in {"runner", "cli"}
assert (PROJECT_ROOT / "dataset").exists(), f"Root proyek tidak ditemukan dari {Path.cwd()}"
for d in (VOL_DIR, MASK_DIR, XFM_DIR, META_DIR, FIG_DIR, TEMPLATE_DIR, WORK_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ITK threads :", os.environ["ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS"])

### CELL 2 — Cek lingkungan

**Tujuan:** mendeteksi (tidak mengasumsikan) versi Python dan library, antspyx, HD-BET/SynthStrip, torch, dan GPU, baik di kernel utama maupun di venv HD-BET. Jika tool wajib tidak ada, cell berhenti dengan instruksi instalasi. **Output:** `ENV` (dict), tabel ketersediaan.

In [ ]:
def pkg_version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None


ENV = {"python": sys.version.split()[0], "platform": platform.platform()}
for name in ["nibabel", "numpy", "nilearn", "antspyx", "scipy", "pandas", "matplotlib", "torch"]:
    ENV[name] = pkg_version(name)
rows = [{"komponen": k, "versi/status": v} for k, v in ENV.items()]

# torch + GPU di kernel utama
try:
    import torch
    ENV["torch_cuda_build"] = torch.version.cuda
    ENV["cuda_available_main"] = bool(torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        ENV["gpu"], ENV["gpu_vram_gib"] = p.name, round(p.total_memory / 2**30, 2)
except ImportError:
    ENV["cuda_available_main"] = False

# venv HD-BET (dicek lewat subprocess, tidak diimport ke kernel)
venv_probe = ("import json, importlib.metadata as m, torch\n"
              "def v(n):\n    try: return m.version(n)\n    except Exception: return None\n"
              "d = {k: v(k) for k in ['hd-bet', 'nnunetv2', 'SimpleITK', 'numpy']}\n"
              "d['torch'] = torch.__version__; d['torch_cuda_build'] = torch.version.cuda\n"
              "d['cuda_available'] = torch.cuda.is_available()\n"
              "d['gpu'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None\n"
              "from HD_BET import paths\n"
              "d['params_dir'] = paths.folder_with_parameter_files; d['params_url'] = paths.ZENODO_DOWNLOAD_URL\n"
              "print(json.dumps(d))")
VENV_ENV = {}
if VENV_PY.exists():
    r = subprocess.run([str(VENV_PY), "-c", venv_probe], capture_output=True, text=True)
    if r.returncode == 0:
        VENV_ENV = json.loads(r.stdout.strip().splitlines()[-1])
    else:
        print("Probe venv gagal:\n", r.stderr[-2000:])
ENV["hdbet_venv"] = VENV_ENV
other_tools = {t: shutil.which(t) for t in ["mri_synthstrip", "bet", "antsRegistration", "N4BiasFieldCorrection", "docker"]}

rows += [{"komponen": "torch CUDA (kernel)", "versi/status": f"{ENV.get('cuda_available_main')} (build {ENV.get('torch_cuda_build')})"},
         {"komponen": "GPU", "versi/status": f"{ENV.get('gpu')} | {ENV.get('gpu_vram_gib')} GiB"},
         {"komponen": "venv HD-BET", "versi/status": str(VENV_DIR.relative_to(PROJECT_ROOT)) if VENV_PY.exists() else "TIDAK ADA"},
         {"komponen": "hd-bet (venv)", "versi/status": VENV_ENV.get("hd-bet")},
         {"komponen": "nnunetv2 (venv)", "versi/status": VENV_ENV.get("nnunetv2")},
         {"komponen": "torch (venv)", "versi/status": f"{VENV_ENV.get('torch')} | CUDA {VENV_ENV.get('cuda_available')} | {VENV_ENV.get('gpu')}"},
         {"komponen": "bobot HD-BET", "versi/status": f"{VENV_ENV.get('params_dir')} | ada: "
          f"{Path(VENV_ENV['params_dir'], 'fold_all', 'checkpoint_final.pth').exists() if VENV_ENV.get('params_dir') else None}"}]
rows += [{"komponen": f"{k} (CLI)", "versi/status": v or "tidak ada"} for k, v in other_tools.items()]
vm = psutil.virtual_memory()
ENV["ram_total_gb"], ENV["ram_available_gb"] = round(vm.total / 2**30, 1), round(vm.available / 2**30, 1)
ENV["psutil"] = pkg_version("psutil")
rows.append({"komponen": "RAM (psutil)", "versi/status": f"total {ENV['ram_total_gb']} GiB | tersedia {ENV['ram_available_gb']} GiB"})
display(pd.DataFrame(rows))

# Tool wajib
if ENV["antspyx"] is None:
    raise RuntimeError("antspyx tidak terpasang di kernel ini. Instal: pip install antspyx==0.6.3 (wheel Windows/Py3.12 tersedia di PyPI).")
if not HDBET_EXE.exists() or not VENV_ENV.get("hd-bet"):
    raise RuntimeError(f"HD-BET tidak ditemukan di {VENV_DIR}. Instal:\n"
                       f"  python -m venv --system-site-packages .venv-hdbet\n"
                       f"  .venv-hdbet\\Scripts\\python -m pip install hd-bet==2.0.1")
if HDBET_MODE == "runner" and not HDBET_RUNNER.exists():
    raise RuntimeError(f"{HDBET_RUNNER} tidak ada.")
if HDBET_DEVICE == "cuda" and not VENV_ENV.get("cuda_available"):
    raise RuntimeError("torch di venv HD-BET tidak bisa memakai CUDA. Jangan pindah ke CPU otomatis; laporkan dan "
                       "putuskan: perbaiki torch CUDA (perlu izin) atau pakai fallback SynthStrip via Docker.")

import ants
ants.config._random_seed = ANTS_RANDOM_SEED
print("antspyx", ants.__version__, "| random seed registrasi:", ants.config._random_seed)

### CELL 2b — Verifikasi CELL 2

In [ ]:
assert ENV["antspyx"] and VENV_ENV.get("hd-bet") and VENV_ENV.get("cuda_available")
assert VENV_ENV.get("torch") == ENV.get("torch"), "torch di venv berbeda dari kernel utama (harusnya dipakai bersama)"
print(f"Kernel: python {ENV['python']} | antspyx {ENV['antspyx']} | nibabel {ENV['nibabel']} | nilearn {ENV['nilearn']}")
print(f"HD-BET {VENV_ENV['hd-bet']} | nnunetv2 {VENV_ENV['nnunetv2']} | torch {VENV_ENV['torch']} | GPU {VENV_ENV['gpu']}")
print("SynthStrip/FSL/ANTs CLI:", {k: bool(v) for k, v in other_tools.items()})

### CELL 3 — Muat data, hash awal, QC, citra identik

**Tujuan:** hash awal input; gabungkan master dengan `smri_qc.csv` (hanya `included_in_final = True`), `folds.csv`, dan `audit_flags.csv`; baca kelompok citra identik dari audit. **Output:** `subjects` (satu baris per subjek yang akan diproses).

In [ ]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


missing_inputs = [p for p in (CLEANED_PATH, SMRI_QC_PATH, FOLDS_PATH, AUDIT_FLAGS_PATH, AUDIT_REPORT_PATH) if not p.exists()]
if missing_inputs:
    raise FileNotFoundError("Input tidak ditemukan: " + ", ".join(str(p.relative_to(PROJECT_ROOT)) for p in missing_inputs)
                            + ". Jalankan notebook/05a_audit_folds.ipynb terlebih dahulu.")
HASH_MASTER_BEFORE, HASH_FOLDS_BEFORE = sha256_file(CLEANED_PATH), sha256_file(FOLDS_PATH)
print("SHA-256 master (awal):", HASH_MASTER_BEFORE)
print("SHA-256 folds  (awal):", HASH_FOLDS_BEFORE)

master = pd.read_csv(CLEANED_PATH)
qc = pd.read_csv(SMRI_QC_PATH)
folds = pd.read_csv(FOLDS_PATH)
aflags = pd.read_csv(AUDIT_FLAGS_PATH, keep_default_na=False, na_values=[""])
qc_in = qc[qc["included_in_final"].astype(bool)][["SUB_ID", "qc_decision", "qc_reason"]]
assert qc_in["SUB_ID"].is_unique and master["SUB_ID"].is_unique

subjects = (master[["SUB_ID", "SITE_ID", "smri_path"]]
            .merge(qc_in, on="SUB_ID", how="inner", validate="one_to_one")
            .merge(folds[["SUB_ID", "label", "group_id"]], on="SUB_ID", how="inner", validate="one_to_one")
            .merge(aflags[["SUB_ID", "smri_voxel_sha256"]], on="SUB_ID", how="left", validate="one_to_one")
            .sort_values("SUB_ID").reset_index(drop=True))
subjects["qc_reason"] = subjects["qc_reason"].fillna("")
subjects["smri_abspath"] = subjects["smri_path"].map(lambda p: DATA_ROOT / Path(str(p)))
subjects["group_size"] = subjects.groupby("group_id")["SUB_ID"].transform("size")
n_excluded = int((~qc["included_in_final"].astype(bool)).sum())
print(f"Subjek diproses: {len(subjects)} | EXCLUDE di smri_qc (tidak diproses): {n_excluded}")
print("qc_decision:", subjects["qc_decision"].value_counts().to_dict())

# Citra identik: dibaca dari audit_flags.csv
ident = subjects[subjects["group_size"] > 1]
IDENTICAL_GROUPS = ident.groupby("group_id")["SUB_ID"].apply(list).to_dict()
req_status = {}
for sid in PILOT_REQUIRED:
    row = subjects[subjects["SUB_ID"] == sid]
    req_status[sid] = ("terverifikasi (grup " + str(int(row["group_id"].iloc[0])) + ")"
                       if len(row) and row["group_size"].iloc[0] > 1 else "tidak terverifikasi")
print("Kelompok citra identik (audit_flags.csv):", IDENTICAL_GROUPS or "-")
print("Status citra identik yang diminta:", req_status)

# image_integrity_flag dari 05b (opsional)
if PHENO_CLEAN_PATH.exists():
    integ = pd.read_csv(PHENO_CLEAN_PATH, usecols=["SUB_ID", "image_integrity_flag"])
    subjects = subjects.merge(integ, on="SUB_ID", how="left", validate="one_to_one")
else:
    subjects["image_integrity_flag"] = np.nan
    print("pheno_clean.csv tidak ada: image_integrity_flag = perlu ditinjau")

### CELL 3b — Verifikasi CELL 3

In [ ]:
assert len(subjects) == int(qc["included_in_final"].astype(bool).sum()) == len(master)
assert subjects["SUB_ID"].is_unique and subjects["label"].isin([0, 1]).all()
assert set(subjects["qc_decision"]) <= {"PASS", "REVIEW"}
print(f"Baris: {len(subjects)} | situs: {subjects['SITE_ID'].nunique()}")
display(pd.crosstab(subjects["SITE_ID"], subjects["qc_decision"]).T)
print("Kelas:", subjects["label"].map({1: "ASD", 0: "TD"}).value_counts().to_dict())

### CELL 4 — Template MNI152 dan box crop tetap

**Tujuan:** memuat template MNI152 dari `nilearn.datasets.load_mni152_template` (versi terpasang dicetak), menyimpannya ke `data/templates/` untuk ANTs, mencetak shape, spasi, dan bounding box mask, lalu menurunkan **box crop tetap** (bbox mask template + margin) yang di-pad ke `TARGET_SHAPE`. Box sama untuk semua subjek.

**Output:** file template, `CROP` (slice), `PAD` (lebar pad), `TEMPLATE_CROP` dan `TEMPLATE_MASK_CROP` (untuk QC).

In [ ]:
from nilearn import datasets as nl_datasets

tmpl_files = {}
for res in sorted({TEMPLATE_RES_REG_MM, TARGET_RES_MM}):
    t_img = nl_datasets.load_mni152_template(resolution=res)
    m_img = nl_datasets.load_mni152_brain_mask(resolution=res)
    assert t_img.shape == m_img.shape and np.allclose(t_img.affine, m_img.affine)
    assert nib.aff2axcodes(t_img.affine) == TARGET_AXCODES, f"Template {res} mm tidak {TARGET_AXCODES}"
    t_data = t_img.get_fdata(dtype=np.float32)
    m_data = (m_img.get_fdata() > 0).astype(np.uint8)
    paths = {"t1": TEMPLATE_DIR / f"mni152_t1_{res}mm.nii.gz", "mask": TEMPLATE_DIR / f"mni152_mask_{res}mm.nii.gz",
             "brain": TEMPLATE_DIR / f"mni152_brain_{res}mm.nii.gz"}
    nib.save(nib.Nifti1Image(t_data, t_img.affine), paths["t1"])
    nib.save(nib.Nifti1Image(m_data, t_img.affine), paths["mask"])
    nib.save(nib.Nifti1Image(t_data * m_data, t_img.affine), paths["brain"])
    idx = np.argwhere(m_data > 0)
    lo, hi = idx.min(0), idx.max(0)
    print(f"[{res} mm] shape {t_img.shape} | voxel {t_img.header.get_zooms()} | axcodes {nib.aff2axcodes(t_img.affine)} | "
          f"bbox mask {lo.tolist()}–{hi.tolist()} (ukuran {(hi - lo + 1).tolist()}) | nilai {t_data.min():.2f}–{t_data.max():.2f}")
    tmpl_files[res] = {"paths": paths, "affine": t_img.affine, "shape": t_img.shape, "data": t_data, "mask": m_data,
                       "bbox": (lo, hi)}

T2 = tmpl_files[TARGET_RES_MM]
lo, hi = T2["bbox"]
CROP, PAD = [], []
for ax in range(3):
    dim, tgt = T2["shape"][ax], TARGET_SHAPE[ax]
    b0, b1 = max(lo[ax] - CROP_MARGIN_VOX, 0), min(hi[ax] + CROP_MARGIN_VOX, dim - 1)
    assert b1 - b0 + 1 <= tgt, f"Sumbu {ax}: bbox+margin ({b1 - b0 + 1}) > target {tgt}"
    if dim >= tgt:                                   # crop tepat tgt, dipusatkan pada bbox
        start = int(np.clip((b0 + b1 + 1) // 2 - tgt // 2, 0, dim - tgt))
        CROP.append(slice(start, start + tgt)); PAD.append((0, 0))
    else:                                            # pakai seluruh sumbu, pad simetris
        CROP.append(slice(0, dim)); extra = tgt - dim
        PAD.append((extra // 2, extra - extra // 2))
    s = CROP[ax]
    assert s.start <= b0 and b1 < s.stop, f"Sumbu {ax}: crop memotong bbox mask template"
CROP, PAD = tuple(CROP), tuple(PAD)


def crop_pad(arr):
    out = np.pad(arr[CROP], PAD, mode="constant", constant_values=0)
    assert out.shape == TARGET_SHAPE, out.shape
    return out


TEMPLATE_CROP = crop_pad(T2["data"] * T2["mask"])
TEMPLATE_MASK_CROP = crop_pad(T2["mask"]).astype(bool)
print("CROP:", [(s.start, s.stop) for s in CROP], "| PAD:", PAD, "| hasil:", TEMPLATE_CROP.shape)

### CELL 4b — Verifikasi CELL 4

In [ ]:
a2 = ants.image_read(str(T2["paths"]["t1"]))
assert np.allclose(a2.numpy(), T2["data"]), "Indeks voxel ANTs ≠ nibabel"
assert TEMPLATE_MASK_CROP.sum() == T2["mask"].sum(), "Crop memotong mask template"
print(f"Template {TARGET_RES_MM} mm: mask {int(T2['mask'].sum())} voxel, utuh setelah crop | "
      f"TEMPLATE_CROP {TEMPLATE_CROP.shape}, rentang {TEMPLATE_CROP.min():.2f}–{TEMPLATE_CROP.max():.2f}")
print("Nilai template di luar mask (frac>0):", float((T2["data"][T2["mask"] == 0] > 0).mean()))

### CELL 5 — Cek header semua citra → `smri_header_check.csv`

**Tujuan:** untuk setiap subjek: file ada, dimensi, voxel size, orientasi, dtype, dan (jika `HEADER_READ_DATA`) NaN/Inf. Hasil di-cache dan dipakai ulang bila path, ukuran, dan mtime file tidak berubah. Anomali hanya ditandai.

**Output:** `smri_header_check.csv`.

In [ ]:
HDR_COLS = ["SUB_ID", "smri_path", "size", "mtime_ns", "exists", "ndim", "shape", "voxel_mm", "axcodes", "dtype",
            "qform_code", "sform_code", "n_nonfinite", "n_negative", "error"]
cache = pd.read_csv(HEADER_CHECK_PATH) if HEADER_CHECK_PATH.exists() else pd.DataFrame(columns=HDR_COLS)
cache_idx = {(int(r.SUB_ID), r.smri_path, int(r.size), int(r.mtime_ns)): r._asdict()
             for r in cache.itertuples(index=False) if isinstance(r.error, float) or not r.error}
hdr_rows, t0 = [], time.time()
for k, r in enumerate(subjects.itertuples(), start=1):
    rec = {c: None for c in HDR_COLS}
    rec.update(SUB_ID=r.SUB_ID, smri_path=r.smri_path, exists=r.smri_abspath.exists(), error="")
    try:
        st = r.smri_abspath.stat()
        rec.update(size=st.st_size, mtime_ns=st.st_mtime_ns)
        hit = cache_idx.get((r.SUB_ID, r.smri_path, st.st_size, st.st_mtime_ns))
        if hit is not None and (not HEADER_READ_DATA or pd.notna(hit.get("n_nonfinite"))):
            rec.update({c: hit[c] for c in HDR_COLS})
        else:
            img = nib.load(str(r.smri_abspath))
            zooms = img.header.get_zooms()[:3]
            rec.update(ndim=img.ndim, shape=str(tuple(img.shape)), voxel_mm=str(tuple(round(float(z), 4) for z in zooms)),
                       axcodes="".join(nib.aff2axcodes(img.affine)), dtype=str(img.get_data_dtype()),
                       qform_code=int(img.header["qform_code"]), sform_code=int(img.header["sform_code"]))
            if HEADER_READ_DATA:
                data = np.asanyarray(img.dataobj)
                rec.update(n_nonfinite=int((~np.isfinite(data)).sum()), n_negative=int((data < 0).sum()))
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"[:300]
        rec["size"] = rec["size"] if rec["size"] is not None else -1
        rec["mtime_ns"] = rec["mtime_ns"] if rec["mtime_ns"] is not None else -1
    hdr_rows.append(rec)
    if k % 200 == 0 or k == len(subjects):
        pd.DataFrame(hdr_rows, columns=HDR_COLS).to_csv(HEADER_CHECK_PATH, index=False)
        print(f"  {k}/{len(subjects)} | {time.time() - t0:.0f} s")

hdr = pd.DataFrame(hdr_rows, columns=HDR_COLS)
vox = hdr["voxel_mm"].dropna().map(eval)
hdr.loc[vox.index, "max_voxel_mm"] = vox.map(max)
hdr.loc[vox.index, "anisotropy"] = vox.map(lambda z: max(z) / min(z))
hdr["flag_missing"] = ~hdr["exists"].astype(bool)
hdr["flag_error"] = hdr["error"].fillna("").ne("")
hdr["flag_not_3d"] = hdr["ndim"].notna() & (hdr["ndim"].astype(float) != 3)
hdr["flag_nonfinite"] = hdr["n_nonfinite"].fillna(0).astype(float) > 0
hdr["flag_anisotropy"] = hdr["anisotropy"].astype(float) > ANISOTROPY_MAX
hdr["flag_voxel_gt"] = hdr["max_voxel_mm"].astype(float) > VOXEL_MAX_MM
hdr["flag_negative"] = hdr["n_negative"].fillna(0).astype(float) > 0
hdr["flag_sform0"] = hdr["sform_code"].fillna(-1).astype(float) == 0
hdr = hdr.merge(subjects[["SUB_ID", "SITE_ID"]], on="SUB_ID")
hdr.to_csv(HEADER_CHECK_PATH, index=False)

### CELL 5b — Verifikasi CELL 5

In [ ]:
flag_cols = [c for c in hdr.columns if c.startswith("flag_")]
assert len(hdr) == len(subjects) and hdr["SUB_ID"].is_unique
print("Jumlah flag:", hdr[flag_cols].sum().to_dict())
print("Orientasi asli:", hdr["axcodes"].value_counts().to_dict())
print("Situs dengan anisotropi/voxel besar:", hdr.loc[hdr["flag_anisotropy"] | hdr["flag_voxel_gt"], "SITE_ID"].value_counts().to_dict())
print("Situs dengan intensitas negatif:", hdr.loc[hdr["flag_negative"], "SITE_ID"].value_counts().to_dict())
blocking = hdr["flag_missing"] | hdr["flag_error"] | hdr["flag_not_3d"] | hdr["flag_nonfinite"]
print(f"Anomali yang menghalangi preprocessing: {int(blocking.sum())}")
if blocking.any():
    display(hdr.loc[blocking, ["SUB_ID", "SITE_ID", "error"] + flag_cols])

### CELL 6 — Fungsi preprocessing

**Tujuan:** fungsi per langkah, dengan setiap langkah di dalam `try/except` dan waktu tiap langkah dicatat:
1. `reorient` → RAS (+ clip intensitas negatif ke 0, dicatat);
2. `n4` → `ants.n4_bias_field_correction`;
3. `skullstrip` → HD-BET (input diorientasikan ke `HDBET_INPUT_AXCODES`, mask dikembalikan ke RAS tanpa interpolasi);
4. `registration` → `ants.registration(type_of_transform="Affine")` otak subjek → otak template 1 mm; matriks disimpan di `data/xfm/`;
5. `resample` → transform yang sama diterapkan ke grid template 2 mm (linear untuk citra, linear + ambang 0.5 untuk mask);
6. `intensity` → clip persentil dan z-score di dalam mask, latar = 0;
7. `crop_save` → crop/pad tetap, simpan `vol/{SUB_ID}.npy` (float32, (1,96,112,96)) dan `mask/{SUB_ID}.npy` (uint8, (1,96,112,96)) secara atomik.

`preprocess_subject(sub_id)` memproses satu subjek. `preprocess_chunk(sub_ids)` memanggil HD-BET **sekali** untuk banyak subjek (model dimuat sekali). Subjek gagal dicatat di `failed_subjects.csv` tanpa menghentikan subjek lain.

**Output:** fungsi; belum ada subjek yang diproses.

In [ ]:
SUBJ = subjects.set_index("SUB_ID")
STEPS = ["reorient", "n4", "skullstrip", "registration", "resample", "intensity", "crop_save"]
FAILED_COLS = ["SUB_ID", "failed_step", "error", "timestamp"]
VRAM_LOG = {}


def out_paths(sid):
    return {"vol": VOL_DIR / f"{sid}.npy", "mask": MASK_DIR / f"{sid}.npy", "xfm": XFM_DIR / f"{sid}_0GenericAffine.mat",
            "meta": META_DIR / f"{sid}.json", "fig": FIG_DIR / f"{sid}.png"}


def output_valid(sid):
    p = out_paths(sid)
    try:
        if not (p["vol"].exists() and p["mask"].exists() and p["meta"].exists()):
            return False
        v, m = np.load(p["vol"], mmap_mode="r"), np.load(p["mask"], mmap_mode="r")
        return (v.shape == (1, *TARGET_SHAPE) and v.dtype == np.float32 and m.shape == (1, *TARGET_SHAPE)
                and bool(np.isfinite(v).all()) and json.loads(p["meta"].read_text())["status"] == "ok")
    except Exception:
        return False


def save_npy_atomic(path, arr):
    tmp = path.with_suffix(".tmp.npy")
    np.save(tmp, arr)
    os.replace(tmp, path)


def reorient_to(img, axcodes):
    t = nib.orientations.ornt_transform(nib.io_orientation(img.affine), nib.orientations.axcodes2ornt(axcodes))
    return img.as_reoriented(t)


class MemoryGuardError(RuntimeError):
    """RAM tersedia di bawah MIN_FREE_RAM_GB: proses dihentikan rapi."""


def ram_available_gb():
    return psutil.virtual_memory().available / 2**30


def check_ram(where):
    avail = ram_available_gb()
    if avail < MIN_FREE_RAM_GB:
        raise MemoryGuardError(f"RAM tersedia {avail:.2f} GiB < {MIN_FREE_RAM_GB} GiB ({where}). "
                               "Tutup aplikasi lain atau kecilkan HDBET_CHUNK, lalu jalankan ulang (resume).")
    return avail


def gpu_used_mib():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=5).stdout
        return float(out.strip().splitlines()[0])
    except Exception:
        return np.nan


def run_hdbet(in_dir, out_dir):
    """Satu panggilan HD-BET untuk semua .nii.gz di in_dir, sambil memantau VRAM dan RAM.
    Gagal -> RuntimeError; RAM di bawah ambang -> proses dibunuh, MemoryGuardError. Tidak ada fallback."""
    if HDBET_MODE == "runner":
        cmd = [str(VENV_PY), str(HDBET_RUNNER), "-i", str(in_dir), "-o", str(out_dir), "--device", HDBET_DEVICE,
               "--tta", str(int(HDBET_TTA)), "--np_preprocessing", str(HDBET_NP_PREPROCESSING),
               "--np_export", str(HDBET_NP_EXPORT)]
    else:
        cmd = [str(HDBET_EXE), "-i", str(in_dir), "-o", str(out_dir), "-device", HDBET_DEVICE,
               "--save_bet_mask", "--no_bet_image"] + ([] if HDBET_TTA else ["--disable_tta"])
    env = dict(os.environ, PYTHONIOENCODING="utf-8")
    check_ram("sebelum HD-BET")
    vram0, vram, ram = gpu_used_mib(), [], []
    log_path = Path(out_dir) / "hdbet_stdout.log"
    t0 = time.time()
    with open(log_path, "w", encoding="utf-8") as log:
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
        killed = None
        while proc.poll() is None:
            vram.append(gpu_used_mib()); ram.append(ram_available_gb())
            if ram[-1] < MIN_FREE_RAM_GB:
                proc.kill(); proc.wait()
                killed = ram[-1]
                break
            time.sleep(0.5)
    dt = time.time() - t0
    out_text = log_path.read_text(encoding="utf-8", errors="replace")
    info = {"seconds": dt, "vram_baseline_mib": vram0, "vram_peak_mib": float(np.nanmax(vram)) if vram else np.nan,
            "ram_min_available_gb": float(min(ram)) if ram else np.nan, "returncode": proc.returncode,
            "mode": HDBET_MODE, "cmd": " ".join(cmd)}
    for line in out_text.splitlines():
        if line.startswith("HDBET_RUNNER_SUMMARY "):
            info["runner"] = json.loads(line[len("HDBET_RUNNER_SUMMARY "):])
    if killed is not None:
        raise MemoryGuardError(f"HD-BET dihentikan: RAM tersedia {killed:.2f} GiB < {MIN_FREE_RAM_GB} GiB. "
                               "Tidak ada fallback otomatis; kecilkan HDBET_CHUNK atau bebaskan RAM, lalu resume.")
    if proc.returncode != 0:
        raise RuntimeError(f"HD-BET gagal (kode {proc.returncode}):\n{out_text[-3000:]}")
    return info


def stage1(sid, wd, rec):
    """reorient + N4, lalu tulis input HD-BET."""
    t = time.time()
    img = nib.load(str(SUBJ.at[sid, "smri_abspath"]))
    if img.ndim == 4 and img.shape[3] == 1:
        img = img.slicer[..., 0]
    assert img.ndim == 3, f"bukan 3D: {img.shape}"
    can = nib.as_closest_canonical(img)
    data = can.get_fdata(dtype=np.float32)
    assert np.isfinite(data).all(), "voxel non-finite"
    rec["n_negative_clipped"] = int((data < 0).sum()) if CLIP_NEGATIVE_BEFORE_N4 else 0
    if CLIP_NEGATIVE_BEFORE_N4:
        data = np.clip(data, 0, None)
    ras = nib.Nifti1Image(data, can.affine)
    ras.set_qform(can.affine, code=1); ras.set_sform(can.affine, code=1)
    nib.save(ras, wd / "ras.nii.gz")
    rec["native_shape_ras"] = list(ras.shape)
    rec["native_voxel_mm"] = [float(z) for z in ras.header.get_zooms()[:3]]
    rec["orig_axcodes"] = "".join(nib.aff2axcodes(img.affine))
    rec["time_reorient"] = time.time() - t

    t = time.time()
    n4 = ants.n4_bias_field_correction(ants.image_read(str(wd / "ras.nii.gz")), **N4_PARAMS)
    ants.image_write(n4, str(wd / "n4.nii.gz"))
    rec["time_n4"] = time.time() - t

    n4_nib = nib.load(str(wd / "n4.nii.gz"))
    assert np.allclose(n4_nib.affine, ras.affine, atol=1e-4), "affine N4 ≠ RAS"
    hd_in = reorient_to(n4_nib, HDBET_INPUT_AXCODES)
    return hd_in


def stage2(sid, wd, mask_hd_path, rec):
    """mask -> RAS, registrasi afin, resample 2 mm, intensitas, crop/pad, simpan."""
    t = time.time()
    n4_nib = nib.load(str(wd / "n4.nii.gz"))
    mask_hd = nib.load(str(mask_hd_path))
    mask_ras = nib.as_closest_canonical(mask_hd)
    assert mask_ras.shape == n4_nib.shape and np.allclose(mask_ras.affine, n4_nib.affine, atol=1e-3), "mask HD-BET tidak sejajar"
    m = (np.asanyarray(mask_ras.dataobj) > 0).astype(np.uint8)
    assert m.sum() > 0, "mask HD-BET kosong"
    nib.save(nib.Nifti1Image(m, n4_nib.affine), wd / "mask_ras.nii.gz")
    brain = n4_nib.get_fdata(dtype=np.float32) * m
    nib.save(nib.Nifti1Image(brain, n4_nib.affine), wd / "brain_ras.nii.gz")
    rec["brain_volume_native_ml"] = float(m.sum() * np.prod(n4_nib.header.get_zooms()[:3]) / 1000)
    rec["time_skullstrip_post"] = time.time() - t

    t = time.time()
    fixed = ants.image_read(str(tmpl_files[TEMPLATE_RES_REG_MM]["paths"]["brain"]))
    moving = ants.image_read(str(wd / "brain_ras.nii.gz"))
    reg = ants.registration(fixed=fixed, moving=moving, type_of_transform=REG_TRANSFORM, outprefix=str(wd / "reg_"))
    xfm = reg["fwdtransforms"]
    assert len(xfm) == 1 and xfm[0].endswith(".mat"), f"transform tak terduga: {xfm}"
    shutil.copyfile(xfm[0], out_paths(sid)["xfm"])
    rec["time_registration"] = time.time() - t

    t = time.time()
    grid = ants.image_read(str(T2["paths"]["t1"]))
    vol = ants.apply_transforms(fixed=grid, moving=moving, transformlist=xfm, interpolator="linear").numpy()
    mk = ants.apply_transforms(fixed=grid, moving=ants.image_read(str(wd / "mask_ras.nii.gz")),
                               transformlist=xfm, interpolator="linear").numpy() > MASK_THRESHOLD
    rec["time_resample"] = time.time() - t

    t = time.time()
    lo_p, hi_p = np.percentile(vol[mk], CLIP_PERCENTILES)
    v = np.clip(vol, lo_p, hi_p)
    mu, sd = float(v[mk].mean()), float(v[mk].std())
    assert sd > 0, "std di dalam mask = 0"
    v = (v - mu) / sd
    v[~mk] = 0
    rec.update(clip_low=float(lo_p), clip_high=float(hi_p), zscore_mean=mu, zscore_std=sd)
    rec["time_intensity"] = time.time() - t

    t = time.time()
    v_c = crop_pad(v).astype(np.float32)[None]
    m_c = crop_pad(mk.astype(np.uint8))[None]
    assert np.isfinite(v_c).all()
    p = out_paths(sid)
    save_npy_atomic(p["vol"], v_c)
    save_npy_atomic(p["mask"], m_c)
    rec["time_crop_save"] = time.time() - t
    rec.update(final_shape=list(v_c.shape), vol_min=float(v_c.min()), vol_max=float(v_c.max()),
               vol_mean=float(v_c.mean()), vol_std=float(v_c.std()),
               vol_mean_in_mask=float(v_c[m_c.astype(bool)].mean()), vol_std_in_mask=float(v_c[m_c.astype(bool)].std()),
               mask_voxels=int(m_c.sum()))
    if SAVE_QC_FIGURE:
        save_qc_figure(sid, wd, v_c[0], m_c[0])
    return rec


def mid_slices(a):
    x, y, z = (s // 2 for s in a.shape)
    return [np.rot90(a[x, :, :]), np.rot90(a[:, y, :]), np.rot90(a[:, :, z])]


def save_qc_figure(sid, wd, vol, mask, show=False):
    ras = nib.load(str(wd / "ras.nii.gz")).get_fdata(dtype=np.float32)
    n4 = nib.load(str(wd / "n4.nii.gz")).get_fdata(dtype=np.float32)
    mras = nib.load(str(wd / "mask_ras.nii.gz")).get_fdata() > 0
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    titles = ["Asli (RAS)", "N4 + mask HD-BET", "Hasil (MNI 2 mm, crop) + kontur mask template"]
    for j, (a, b) in enumerate(zip(mid_slices(ras), mid_slices(n4))):
        axes[0, j].imshow(a, cmap="gray"); axes[1, j].imshow(b, cmap="gray")
    for j, mk in enumerate(mid_slices(mras.astype(float))):
        axes[1, j].contour(mk, levels=[0.5], colors="r", linewidths=0.6)
    for j, (a, t) in enumerate(zip(mid_slices(vol), mid_slices(TEMPLATE_MASK_CROP.astype(float)))):
        axes[2, j].imshow(a, cmap="gray", vmin=-3, vmax=3)
        axes[2, j].contour(t, levels=[0.5], colors="c", linewidths=0.6)
    for i in range(3):
        axes[i, 0].set_ylabel(titles[i], fontsize=8)
        for j in range(3):
            axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
    meta = SUBJ.loc[sid]
    fig.suptitle(f"SUB_ID {sid} | {meta['SITE_ID']} | {'ASD' if meta['label'] == 1 else 'TD'} | QC {meta['qc_decision']}")
    fig.tight_layout()
    fig.savefig(out_paths(sid)["fig"], dpi=80)
    if show:
        plt.show()
    plt.close(fig)


def record_failure(sid, step, err):
    rows = pd.read_csv(FAILED_PATH).to_dict("records") if FAILED_PATH.exists() else []
    rows = [r for r in rows if int(r["SUB_ID"]) != int(sid)]
    rows.append({"SUB_ID": int(sid), "failed_step": step, "error": str(err)[:1000],
                 "timestamp": datetime.datetime.now().isoformat(timespec="seconds")})
    pd.DataFrame(rows, columns=FAILED_COLS).to_csv(FAILED_PATH, index=False)


def clear_failure(sid):
    if FAILED_PATH.exists():
        f = pd.read_csv(FAILED_PATH)
        f[f["SUB_ID"] != int(sid)].to_csv(FAILED_PATH, index=False)


def preprocess_chunk(sub_ids, skip_valid=True, raise_on_hdbet_error=False):
    """Proses beberapa subjek; HD-BET dipanggil sekali untuk semua. Mengembalikan list record."""
    records, hd_inputs = {}, {}
    chunk_id = f"chunk_{min(sub_ids)}_{len(sub_ids)}"
    hd_in_dir, hd_out_dir = WORK_DIR / chunk_id / "hdbet_in", WORK_DIR / chunk_id / "hdbet_out"
    shutil.rmtree(WORK_DIR / chunk_id, ignore_errors=True)
    hd_in_dir.mkdir(parents=True); hd_out_dir.mkdir(parents=True)

    for sid in sub_ids:
        sid = int(sid)
        if skip_valid and output_valid(sid):
            records[sid] = {"SUB_ID": sid, "status": "skipped_valid"}
            continue
        check_ram(f"sebelum subjek {sid}")
        wd = WORK_DIR / str(sid)
        shutil.rmtree(wd, ignore_errors=True); wd.mkdir(parents=True)
        rec = {"SUB_ID": sid, "status": "running", "failed_step": None, "error": None,
               "started": datetime.datetime.now().isoformat(timespec="seconds")}
        step = "reorient"
        try:
            hd_img = stage1(sid, wd, rec)
            step = "skullstrip"
            nib.save(hd_img, hd_in_dir / f"{sid}.nii.gz")
            hd_inputs[sid] = wd
        except Exception as e:
            step = "n4" if "time_reorient" in rec and step == "reorient" else step
            rec.update(status="failed", failed_step=step, error=f"{type(e).__name__}: {e}")
            record_failure(sid, step, rec["error"])
        records[sid] = rec

    if hd_inputs:
        try:
            info = run_hdbet(hd_in_dir, hd_out_dir)
            VRAM_LOG[chunk_id] = info
            per_subject = info["seconds"] / len(hd_inputs)
        except Exception as e:
            for sid in hd_inputs:
                records[sid].update(status="failed", failed_step="skullstrip", error=str(e)[:1000])
                record_failure(sid, "skullstrip", records[sid]["error"])
                out_paths(sid)["meta"].write_text(json.dumps(records[sid], indent=1, default=str))
            if raise_on_hdbet_error or isinstance(e, MemoryGuardError):
                raise
            hd_inputs = {}

    for sid, wd in hd_inputs.items():
        rec = records[sid]
        rec["time_skullstrip"] = per_subject
        rec["hdbet_chunk_seconds"], rec["hdbet_chunk_n"] = info["seconds"], len(sub_ids)
        rec["vram_peak_mib"], rec["vram_baseline_mib"] = info["vram_peak_mib"], info["vram_baseline_mib"]
        rec["ram_min_available_gb"], rec["hdbet_mode"] = info["ram_min_available_gb"], info["mode"]
        rec["hdbet_model_load_s"] = info.get("runner", {}).get("model_load_seconds")
        mask_path = hd_out_dir / f"{sid}_bet.nii.gz"
        try:
            if not mask_path.exists():
                raise FileNotFoundError(f"mask HD-BET tidak ada: {mask_path.name}")
            stage2(sid, wd, mask_path, rec)
            rec["status"] = "ok"
            clear_failure(sid)
        except Exception as e:
            if "time_skullstrip_post" not in rec:
                failed = "skullstrip"
            else:
                failed = next((s for s in STEPS[3:] if f"time_{s}" not in rec), "crop_save")
            rec.update(status="failed", failed_step=failed, error=f"{type(e).__name__}: {e}")
            rec["traceback"] = traceback.format_exc()[-2000:]
            record_failure(sid, failed, rec["error"])
        rec["time_total"] = sum(v for k, v in rec.items() if k.startswith("time_") and k != "time_total")
        out_paths(sid)["meta"].write_text(json.dumps(rec, indent=1, default=str))
        if not KEEP_WORK_FILES and rec["status"] == "ok":
            shutil.rmtree(wd, ignore_errors=True)
    if not KEEP_WORK_FILES:
        shutil.rmtree(WORK_DIR / chunk_id, ignore_errors=True)
    for sid, rec in records.items():
        if rec["status"] == "failed" and not out_paths(sid)["meta"].exists():
            out_paths(sid)["meta"].write_text(json.dumps(rec, indent=1, default=str))
    return [records[int(s)] for s in sub_ids]


def preprocess_subject(sub_id, skip_valid=True, raise_on_hdbet_error=False):
    return preprocess_chunk([sub_id], skip_valid=skip_valid, raise_on_hdbet_error=raise_on_hdbet_error)[0]

### CELL 6b — Verifikasi CELL 6

In [ ]:
z = np.zeros(T2["shape"], dtype=np.float32)
assert crop_pad(z).shape == TARGET_SHAPE
assert callable(preprocess_subject) and callable(preprocess_chunk)
assert HDBET_EXE.exists() and (HDBET_MODE != "runner" or HDBET_RUNNER.exists())
print("Langkah:", STEPS)
print(f"Output per subjek: vol (1,{TARGET_SHAPE}) float32, mask uint8, xfm .mat, meta .json, figure .png")
print("Subjek dengan output valid saat ini:", sum(output_valid(s) for s in subjects["SUB_ID"]))

### CELL 7 — Validasi runner HD-BET vs CLI

**Tujuan:** memastikan `src/hdbet_runner.py` (model dimuat sekali, 1 proses preprocessing/export) menghasilkan mask yang sama dengan CLI `hd-bet` pada subjek test. Mask CLI acuan diambil dari run sebelumnya (`work/chunk_<test>_1/hdbet_out`) jika ada; jika tidak, CLI dijalankan sekali. Lolos jika Dice ≥ `VALIDATION_DICE_MIN`; jika tidak, notebook **berhenti** (fallback yang disetujui: `HDBET_MODE="cli"` dengan `HDBET_CHUNK=1`).

**Output:** `work/hdbet_validation/` (mask CLI dan runner), `VALIDATION`.

In [ ]:
test_sid = TEST_SUB_ID or int(subjects.loc[subjects["qc_decision"] == "PASS", "SUB_ID"].min())
val_dir = WORK_DIR / "hdbet_validation"
(val_dir / "in").mkdir(parents=True, exist_ok=True)
for sub in ("cli", "runner"):
    (val_dir / sub).mkdir(exist_ok=True)
val_in = val_dir / "in" / f"{test_sid}.nii.gz"
prev_chunk = WORK_DIR / f"chunk_{test_sid}_1"
if not val_in.exists():
    if (prev_chunk / "hdbet_in" / f"{test_sid}.nii.gz").exists():
        shutil.copyfile(prev_chunk / "hdbet_in" / f"{test_sid}.nii.gz", val_in)
    else:
        wd_tmp = WORK_DIR / f"val_{test_sid}"; wd_tmp.mkdir(exist_ok=True)
        nib.save(stage1(test_sid, wd_tmp, {}), val_in)
cli_mask = val_dir / "cli" / f"{test_sid}_bet.nii.gz"
cli_source = "run CLI sebelumnya (test yang lolos)"
if not cli_mask.exists():
    if (prev_chunk / "hdbet_out" / f"{test_sid}_bet.nii.gz").exists():
        shutil.copyfile(prev_chunk / "hdbet_out" / f"{test_sid}_bet.nii.gz", cli_mask)
    else:
        cli_source = "CLI dijalankan sekarang"
        _mode = HDBET_MODE; HDBET_MODE = "cli"
        try:
            run_hdbet(val_dir / "in", val_dir / "cli")
        finally:
            HDBET_MODE = _mode
assert nib.aff2axcodes(nib.load(str(val_in)).affine) == HDBET_INPUT_AXCODES

_mode = HDBET_MODE; HDBET_MODE = "runner"
try:
    val_info = run_hdbet(val_dir / "in", val_dir / "runner")
finally:
    HDBET_MODE = _mode
m_cli = np.asanyarray(nib.load(str(cli_mask)).dataobj) > 0
m_run_img = nib.load(str(val_dir / "runner" / f"{test_sid}_bet.nii.gz"))
m_run = np.asanyarray(m_run_img.dataobj) > 0
assert m_cli.shape == m_run.shape and np.allclose(nib.load(str(cli_mask)).affine, m_run_img.affine, atol=1e-4)
dice = float(2 * (m_cli & m_run).sum() / (m_cli.sum() + m_run.sum()))
VALIDATION = {"SUB_ID": test_sid, "cli_source": cli_source, "dice": dice, "bitwise_identical": bool(np.array_equal(m_cli, m_run)),
              "n_voxel_cli": int(m_cli.sum()), "n_voxel_runner": int(m_run.sum()), "n_voxel_diff": int((m_cli ^ m_run).sum()),
              "runner_seconds": val_info["seconds"], "runner_model_load_s": val_info.get("runner", {}).get("model_load_seconds"),
              "runner_predict_s": val_info.get("runner", {}).get("predict_seconds"),
              "vram_peak_mib": val_info["vram_peak_mib"], "ram_min_available_gb": val_info["ram_min_available_gb"]}
print(json.dumps(VALIDATION, indent=1))
if not (VALIDATION["bitwise_identical"] or dice >= VALIDATION_DICE_MIN):
    raise RuntimeError(f"Validasi runner GAGAL: Dice {dice:.5f} < {VALIDATION_DICE_MIN}. Pilot tidak dijalankan. "
                       "Fallback yang disetujui: HDBET_MODE='cli' dan HDBET_CHUNK=1.")

### CELL 7b — Verifikasi CELL 7

In [ ]:
assert VALIDATION["bitwise_identical"] or VALIDATION["dice"] >= VALIDATION_DICE_MIN
print(f"Runner vs CLI ({test_sid}): Dice {VALIDATION['dice']:.6f} | bitwise identik: {VALIDATION['bitwise_identical']} | "
      f"selisih {VALIDATION['n_voxel_diff']} voxel | runner {VALIDATION['runner_seconds']:.0f} s "
      f"(muat model {VALIDATION['runner_model_load_s']}, prediksi {VALIDATION['runner_predict_s']}) | "
      f"VRAM puncak {VALIDATION['vram_peak_mib']:.0f} MiB | RAM tersedia min {VALIDATION['ram_min_available_gb']:.2f} GiB")

### CELL 7c — Test 1 subjek

**Tujuan:** menjalankan seluruh pipeline pada satu subjek dengan HD-BET di GPU, lalu menampilkan overlay (asli, N4 + mask, hasil registrasi di atas template), shape, min/max/mean/std, waktu per langkah, dan VRAM puncak (`nvidia-smi`, termasuk pemakaian proses lain). Jika HD-BET gagal, notebook **berhenti** (tidak pindah ke CPU atau metode lain).

**Output:** output subjek test di `vol/`, `mask/`, `xfm/`, `vol_meta/`, `qc_figures/`.

In [ ]:
print("Subjek test:", test_sid, dict(SUBJ.loc[test_sid, ["SITE_ID", "label", "qc_decision"]]))
print("Header asli:", hdr.loc[hdr["SUB_ID"] == test_sid, ["shape", "voxel_mm", "axcodes", "dtype"]].to_dict("records")[0])
t0 = time.time()
test_rec = preprocess_subject(test_sid, skip_valid=False, raise_on_hdbet_error=True)
TEST_WALL = time.time() - t0
if test_rec["status"] != "ok":
    raise RuntimeError(f"Test gagal pada langkah {test_rec['failed_step']}: {test_rec['error']}\n{test_rec.get('traceback', '')}")

# Cek orientasi input HD-BET (syarat README)
hd_in_file = next((WORK_DIR / f"chunk_{test_sid}_1" / "hdbet_in").glob("*.nii.gz"))
print("Orientasi input HD-BET:", nib.aff2axcodes(nib.load(str(hd_in_file)).affine), "| diharapkan:", HDBET_INPUT_AXCODES)
save_qc_figure(test_sid, WORK_DIR / str(test_sid), np.load(out_paths(test_sid)["vol"])[0],
               np.load(out_paths(test_sid)["mask"])[0], show=True)
timing = {s: round(test_rec.get(f"time_{s}", np.nan), 2) for s in STEPS + ["skullstrip_post"]}
print("Waktu per langkah (s):", timing, "| total:", round(test_rec["time_total"], 1), "| wall:", round(TEST_WALL, 1))
print(f"VRAM: baseline {test_rec['vram_baseline_mib']:.0f} MiB | puncak {test_rec['vram_peak_mib']:.0f} MiB "
      f"| dari {ENV.get('gpu_vram_gib')} GiB | RAM tersedia min {test_rec['ram_min_available_gb']:.2f} GiB | mode {test_rec['hdbet_mode']}")
print({k: test_rec[k] for k in ["final_shape", "vol_min", "vol_max", "vol_mean", "vol_std", "vol_mean_in_mask",
                                "vol_std_in_mask", "brain_volume_native_ml", "n_negative_clipped"]})

### CELL 7d — Verifikasi CELL 7c

In [ ]:
v_t = np.load(out_paths(test_sid)["vol"]); m_t = np.load(out_paths(test_sid)["mask"])
assert v_t.shape == (1, *TARGET_SHAPE) and v_t.dtype == np.float32 and np.isfinite(v_t).all()
assert m_t.shape == (1, *TARGET_SHAPE) and set(np.unique(m_t)) <= {0, 1}
assert abs(v_t[m_t.astype(bool)].mean()) < 0.05 and abs(v_t[m_t.astype(bool)].std() - 1) < 0.05
assert (v_t[~m_t.astype(bool)] == 0).all(), "latar bukan 0"
assert nib.aff2axcodes(nib.load(str(hd_in_file)).affine) == HDBET_INPUT_AXCODES
print(f"Test {test_sid}: OK | shape {v_t.shape} | in-mask mean {v_t[m_t.astype(bool)].mean():.3f} std "
      f"{v_t[m_t.astype(bool)].std():.3f} | mask {int(m_t.sum())} voxel (template {int(TEMPLATE_MASK_CROP.sum())})")

### CELL 8 — Pilot 20 subjek

**Tujuan:** memilih 20 subjek secara acak (seed tetap) dengan stratifikasi situs dan label, **wajib** menyertakan `PILOT_REQUIRED` (citra identik), lalu memprosesnya dengan satu panggilan HD-BET per `HDBET_CHUNK` subjek. Resume aktif (subjek dengan output valid dilewati). Figure QC disimpan per subjek. Waktu rata-rata per subjek dipakai untuk mengestimasi total 1.004 subjek.

**Output:** `PILOT_IDS`, output pilot, estimasi waktu.

In [ ]:
rng = np.random.default_rng(SEED)
pilot = [s for s in PILOT_REQUIRED if s in set(subjects["SUB_ID"])]
missing_req = [s for s in PILOT_REQUIRED if s not in pilot]
if missing_req:
    print("PILOT_REQUIRED tidak ada di data (tidak terverifikasi):", missing_req)
sites = list(rng.permutation(sorted(subjects["SITE_ID"].unique())))
lab_cycle = 0
while len(pilot) < PILOT_N:
    for site in sites:
        if len(pilot) >= PILOT_N:
            break
        lab = lab_cycle % 2
        pool = subjects[(subjects["SITE_ID"] == site) & (subjects["label"] == lab) & ~subjects["SUB_ID"].isin(pilot)
                        & ~subjects["group_id"].isin(subjects.loc[subjects["SUB_ID"].isin(pilot), "group_id"])]
        if len(pool):
            pilot.append(int(rng.choice(pool["SUB_ID"].values)))
            lab_cycle += 1
PILOT_IDS = sorted(pilot)
pilot_tbl = subjects[subjects["SUB_ID"].isin(PILOT_IDS)][["SUB_ID", "SITE_ID", "label", "qc_decision", "group_size"]]
display(pilot_tbl)
print("Situs:", pilot_tbl["SITE_ID"].nunique(), "| label:", pilot_tbl["label"].value_counts().to_dict(),
      "| QC:", pilot_tbl["qc_decision"].value_counts().to_dict())

def verify_input_hashes(tag):
    hm, hf = sha256_file(CLEANED_PATH), sha256_file(FOLDS_PATH)
    print(f"[{tag}] hash master {'tidak berubah' if hm == HASH_MASTER_BEFORE else 'BERUBAH'} | "
          f"hash folds {'tidak berubah' if hf == HASH_FOLDS_BEFORE else 'BERUBAH'}")
    assert hm == HASH_MASTER_BEFORE and hf == HASH_FOLDS_BEFORE, "Input read-only berubah!"


t0 = time.time()
PILOT_RECORDS, RAM_LOG, PILOT_STOPPED = [], [], None
try:
    for i in range(0, len(PILOT_IDS), HDBET_CHUNK):
        chunk = PILOT_IDS[i:i + HDBET_CHUNK]
        before = ram_available_gb()
        try:
            PILOT_RECORDS += preprocess_chunk(chunk, skip_valid=True)
        except MemoryGuardError as e:
            PILOT_STOPPED = str(e)
            print("PILOT DIHENTIKAN:", e)
            break
        finally:
            RAM_LOG.append({"chunk": f"{chunk[0]}..{chunk[-1]} (n={len(chunk)})", "ram_before_gb": round(before, 2),
                            "ram_after_gb": round(ram_available_gb(), 2),
                            "ram_min_during_hdbet_gb": round(list(VRAM_LOG.values())[-1]["ram_min_available_gb"], 2) if VRAM_LOG else None})
finally:
    PILOT_WALL = time.time() - t0
    verify_input_hashes("pilot")
display(pd.DataFrame(RAM_LOG))

pilot_meta = pd.DataFrame([json.loads(out_paths(s)["meta"].read_text()) for s in PILOT_IDS if out_paths(s)["meta"].exists()])
ok = pilot_meta[pilot_meta["status"] == "ok"] if "status" in pilot_meta else pilot_meta.iloc[0:0]
time_cols = [c for c in [f"time_{s}" for s in STEPS + ["skullstrip_post"]] + ["time_total"] if c in ok.columns]
n_processed_now = sum(r["status"] == "ok" for r in PILOT_RECORDS)
if len(ok) and time_cols:
    print("\nWaktu per langkah (s), subjek pilot ok:")
    display(ok[time_cols].describe().loc[["mean", "std", "min", "max"]].round(2))
    mean_total = float(ok["time_total"].mean())
else:
    print("\nTidak ada subjek pilot yang ok: ringkasan waktu tidak tersedia.")
    mean_total = float("nan")
n_remaining = len(subjects) - sum(output_valid(s) for s in subjects["SUB_ID"])
print(f"Wall time pilot: {PILOT_WALL / 60:.1f} menit untuk {n_processed_now} subjek diproses sekarang "
      f"({PILOT_WALL / max(n_processed_now, 1):.1f} s/subjek termasuk overhead)")
print(f"Estimasi total 1.004 subjek: {mean_total * len(subjects) / 3600:.1f} jam (rata-rata langkah) | "
      f"{PILOT_WALL / max(n_processed_now, 1) * len(subjects) / 3600:.1f} jam (wall pilot) | sisa subjek: {n_remaining}")
print("HD-BET per chunk:", {k: f"{v['mode']}: {v['seconds']:.0f} s, VRAM puncak {v['vram_peak_mib']:.0f} MiB, "
                               f"RAM min {v['ram_min_available_gb']:.2f} GiB" for k, v in VRAM_LOG.items()})

### CELL 8b — Verifikasi CELL 8

In [ ]:
assert len(PILOT_IDS) == PILOT_N and set(PILOT_REQUIRED) <= set(PILOT_IDS)
if PILOT_STOPPED:
    raise RuntimeError("Pilot dihentikan oleh pengaman RAM: " + PILOT_STOPPED)
failed_pilot = pilot_meta[pilot_meta["status"] != "ok"] if "status" in pilot_meta else pilot_meta
print(f"Pilot: {len(PILOT_IDS)} subjek | ok: {len(ok)} | gagal: {len(failed_pilot)}")
if len(failed_pilot):
    display(failed_pilot[["SUB_ID", "failed_step", "error"]])
for s in ok["SUB_ID"]:
    assert output_valid(int(s)), f"output {s} tidak valid"
print("Figure QC:", sum(out_paths(s)["fig"].exists() for s in PILOT_IDS), "/", len(PILOT_IDS))

### CELL 9 — QC otomatis → `qc_report.csv` + galeri

**Tujuan:** untuk setiap subjek yang sudah diproses: korelasi (NCC) dengan template di dalam mask template, rasio volume mask terhadap mask template, fraksi voxel nol, NaN/Inf, shape akhir, z-score outlier (global bila n ≥ `MIN_N_GLOBAL_Z`, per situs bila n situs ≥ `MIN_N_SITE_Z`), plus `qc_decision`/`qc_reason` dari `smri_qc.csv` dan penanda citra identik. Tidak ada subjek yang dibuang; kolom `pass_only` disediakan untuk perbandingan pass_only vs pass_plus_review.

**Output:** `qc_report.csv`, `qc_figures/gallery_top_flags.png`, `qc_figures/gallery_random.png`.

In [ ]:
TMPL_Z = np.where(TEMPLATE_MASK_CROP, (TEMPLATE_CROP - TEMPLATE_CROP[TEMPLATE_MASK_CROP].mean())
                  / TEMPLATE_CROP[TEMPLATE_MASK_CROP].std(), 0)


def qc_metrics(sid):
    v, m = np.load(out_paths(sid)["vol"])[0], np.load(out_paths(sid)["mask"])[0].astype(bool)
    meta = json.loads(out_paths(sid)["meta"].read_text())
    return {"SUB_ID": sid, "status": meta["status"], "final_shape": str(v.shape), "n_nonfinite": int((~np.isfinite(v)).sum()),
            "frac_zero": float((v == 0).mean()),
            "ncc_template": float(np.corrcoef(v[TEMPLATE_MASK_CROP], TMPL_Z[TEMPLATE_MASK_CROP])[0, 1]),
            "mask_ratio_template": float(m.sum() / TEMPLATE_MASK_CROP.sum()),
            "dice_mask_template": float(2 * (m & TEMPLATE_MASK_CROP).sum() / (m.sum() + TEMPLATE_MASK_CROP.sum())),
            "brain_volume_native_ml": meta.get("brain_volume_native_ml"), "time_total_s": meta.get("time_total")}


def build_qc_report():
    done = [s for s in subjects["SUB_ID"] if output_valid(int(s))]
    rep = pd.DataFrame([qc_metrics(int(s)) for s in done])
    if rep.empty:
        return rep
    rep = rep.merge(subjects[["SUB_ID", "SITE_ID", "label", "qc_decision", "qc_reason", "group_id", "group_size",
                              "image_integrity_flag"]], on="SUB_ID")
    rep["flag_image_identical"] = rep["group_size"] > 1
    rep["identical_group_id"] = np.where(rep["flag_image_identical"], rep["group_id"], np.nan)
    rep["pass_only"] = rep["qc_decision"].eq("PASS")
    metrics = ["ncc_template", "mask_ratio_template", "frac_zero"]
    for c in metrics:
        if len(rep) >= MIN_N_GLOBAL_Z:
            rep[f"z_{c}"] = (rep[c] - rep[c].mean()) / rep[c].std(ddof=0)
        else:
            rep[f"z_{c}"] = np.nan
        g = rep.groupby("SITE_ID")[c]
        n_site = g.transform("size")
        zs = (rep[c] - g.transform("mean")) / g.transform(lambda s: s.std(ddof=0))
        rep[f"zsite_{c}"] = np.where(n_site >= MIN_N_SITE_Z, zs, np.nan)
    zcols = [c for c in rep.columns if c.startswith("z_")]
    zscols = [c for c in rep.columns if c.startswith("zsite_")]
    rep["flag_outlier_global"] = (rep[zcols].abs() > OUTLIER_Z).any(axis=1)
    rep["flag_outlier_site"] = (rep[zscols].abs() > OUTLIER_Z).any(axis=1)
    rep["flag_hard"] = (rep["n_nonfinite"] > 0) | (rep["final_shape"] != str(TARGET_SHAPE))
    rep["flag_outlier"] = rep["flag_outlier_global"] | rep["flag_outlier_site"] | rep["flag_hard"]
    rep["max_abs_z"] = rep[zcols + zscols].abs().max(axis=1)
    rep["outlier_z_available"] = rep[zcols + zscols].notna().any(axis=1)
    return rep.sort_values("SUB_ID").reset_index(drop=True)


def gallery(ids, path, title):
    n = len(ids)
    cols = 6
    rows_ = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows_, cols, figsize=(cols * 2.2, rows_ * 2.6))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, sid in zip(np.atleast_1d(axes).ravel(), ids):
        v = np.load(out_paths(sid)["vol"])[0]
        ax.imshow(np.rot90(v[:, :, v.shape[2] // 2]), cmap="gray", vmin=-3, vmax=3)
        ax.contour(np.rot90(TEMPLATE_MASK_CROP[:, :, v.shape[2] // 2].astype(float)), levels=[0.5], colors="c", linewidths=0.5)
        r = QC_REPORT.set_index("SUB_ID").loc[sid]
        ax.set_title(f"{sid} {r['SITE_ID']}\nncc {r['ncc_template']:.2f} z {r['max_abs_z']:.1f}", fontsize=7)
    fig.suptitle(title); fig.tight_layout(); fig.savefig(path, dpi=80); plt.close(fig)


QC_REPORT = build_qc_report()
assert len(QC_REPORT), "Belum ada subjek dengan output valid; QC tidak dapat dibuat."
QC_REPORT.to_csv(QC_REPORT_PATH, index=False)
top_ids = QC_REPORT.sort_values("max_abs_z", ascending=False, na_position="last")["SUB_ID"].head(GALLERY_N).tolist()
rand_ids = list(np.random.default_rng(SEED).choice(QC_REPORT["SUB_ID"], size=min(GALLERY_N, len(QC_REPORT)), replace=False))
gallery(top_ids, FIG_DIR / "gallery_top_flags.png", f"{len(top_ids)} subjek |z| tertinggi")
gallery(rand_ids, FIG_DIR / "gallery_random.png", f"{len(rand_ids)} subjek acak")
display(QC_REPORT[["SUB_ID", "SITE_ID", "qc_decision", "ncc_template", "mask_ratio_template", "dice_mask_template",
                   "frac_zero", "brain_volume_native_ml", "max_abs_z", "flag_outlier", "flag_image_identical"]].round(3))

### CELL 9b — Verifikasi CELL 9

In [ ]:
assert QC_REPORT["SUB_ID"].is_unique and set(PILOT_IDS) - set(failed_pilot["SUB_ID"]) <= set(QC_REPORT["SUB_ID"])
assert (QC_REPORT["n_nonfinite"] == 0).all() and (QC_REPORT["final_shape"] == str(TARGET_SHAPE)).all()
print(f"qc_report.csv: {QC_REPORT.shape} | outlier: {int(QC_REPORT['flag_outlier'].sum())} | "
      f"z global tersedia: {QC_REPORT['z_ncc_template'].notna().any()} | z per situs tersedia: "
      f"{QC_REPORT[[c for c in QC_REPORT.columns if c.startswith('zsite_')]].notna().any().any()}")
print("NCC:", QC_REPORT["ncc_template"].describe()[["min", "mean", "max"]].round(3).to_dict(),
      "| rasio mask:", QC_REPORT["mask_ratio_template"].describe()[["min", "mean", "max"]].round(3).to_dict())

### CELL 10 — Cek citra identik

**Tujuan:** subjek dalam kelompok citra identik (dari audit 05a) diproses seperti biasa. Volume hasil preprocessing dibandingkan: assert identik atau selisih maksimum ≤ `IDENTICAL_ATOL`. Selisih dicetak dan ditandai di `qc_report.csv`. Tidak ada yang dihapus.

**Output:** kolom `identical_max_abs_diff`, `identical_within_tol` di `qc_report.csv`.

In [ ]:
QC_REPORT["identical_max_abs_diff"] = np.nan
QC_REPORT["identical_within_tol"] = pd.NA
ident_results = []
for gid, members in IDENTICAL_GROUPS.items():
    done = [s for s in members if output_valid(int(s))]
    if len(done) < 2:
        print(f"Grup {gid}: belum semua anggota diproses ({done} dari {members})")
        continue
    ref = np.load(out_paths(done[0])["vol"])
    for s in done[1:]:
        other = np.load(out_paths(s)["vol"])
        diff = float(np.abs(ref - other).max())
        corr = float(np.corrcoef(ref.ravel(), other.ravel())[0, 1])
        ident_results.append({"group_id": gid, "a": done[0], "b": s, "max_abs_diff": diff,
                              "mean_abs_diff": float(np.abs(ref - other).mean()), "corr": corr,
                              "bitwise_identical": bool(np.array_equal(ref, other)), "within_tol": diff <= IDENTICAL_ATOL})
    for s in done:
        mx = max(r["max_abs_diff"] for r in ident_results if r["group_id"] == gid)
        QC_REPORT.loc[QC_REPORT["SUB_ID"] == s, "identical_max_abs_diff"] = mx
        QC_REPORT.loc[QC_REPORT["SUB_ID"] == s, "identical_within_tol"] = mx <= IDENTICAL_ATOL
ident_df = pd.DataFrame(ident_results)
display(ident_df)
QC_REPORT.to_csv(QC_REPORT_PATH, index=False)

### CELL 10b — Verifikasi CELL 10

In [ ]:
for r in ident_results:
    print(f"Grup {r['group_id']}: {r['a']} vs {r['b']} | max|Δ| {r['max_abs_diff']:.2e} | mean|Δ| {r['mean_abs_diff']:.2e} | "
          f"corr {r['corr']:.6f} | bitwise identik: {r['bitwise_identical']}")
    assert r["within_tol"], (f"Volume citra identik berbeda melebihi toleransi ({r['max_abs_diff']:.2e} > {IDENTICAL_ATOL}). "
                             "Periksa determinisme ANTs/HD-BET sebelum batch penuh.")
print("Citra identik diproses:", len(ident_results), "pasangan | tetap dipertahankan, ditandai di qc_report.csv")

### CELL 11 — Batch penuh (kode saja)

**Tujuan:** memproses seluruh 1.004 subjek. **Tidak dijalankan di notebook ini** (`RUN_FULL_BATCH = False`). Set `RUN_FULL_BATCH = True` lalu jalankan cell ini secara manual.
- Resume: subjek dengan output valid (`output_valid`) dilewati.
- Progress bar (`tqdm`), simpan berkala: output per subjek ditulis langsung; `qc_report.csv` diperbarui tiap chunk.
- Gagal per subjek dicatat di `failed_subjects.csv` (SUB_ID, langkah gagal, pesan error); batch tetap jalan.
- `KEEP_WORK_FILES = False` agar file antara dihapus (hemat disk).

**Output:** seluruh `vol/`, `mask/`, `xfm/`, `vol_meta/`, `qc_figures/`, `qc_report.csv`, `failed_subjects.csv`.

In [ ]:
if RUN_FULL_BATCH:
    from tqdm.auto import tqdm
    KEEP_WORK_FILES = False
    todo = [int(s) for s in subjects["SUB_ID"] if not output_valid(int(s))]
    print(f"Batch penuh: {len(todo)} subjek tersisa dari {len(subjects)}")
    with tqdm(total=len(todo), desc="05c batch") as bar:
        for i in range(0, len(todo), HDBET_CHUNK):
            chunk = todo[i:i + HDBET_CHUNK]
            recs = preprocess_chunk(chunk, skip_valid=True)
            bar.update(len(chunk))
            bar.set_postfix(ok=sum(r["status"] == "ok" for r in recs), gagal=sum(r["status"] == "failed" for r in recs))
            QC_REPORT = build_qc_report()
            QC_REPORT.to_csv(QC_REPORT_PATH, index=False)
    n_failed = len(pd.read_csv(FAILED_PATH)) if FAILED_PATH.exists() else 0
    print(f"Selesai: valid {sum(output_valid(int(s)) for s in subjects['SUB_ID'])} | gagal {n_failed}")
else:
    print("RUN_FULL_BATCH = False: batch penuh tidak dijalankan.")

### CELL 12 — Analisis QC vs label/situs (kode saja)

**Tujuan:** setelah batch penuh, menguji apakah flag outlier terkonsentrasi di situs atau label tertentu (tabel + chi-square). **Tidak membuang** subjek REVIEW maupun yang ditandai; hanya menyediakan perbandingan `pass_only` vs `pass_plus_review`. Dijalankan bila `RUN_QC_ANALYSIS = True`.

**Output:** tabel di notebook.

In [ ]:
def qc_vs_label_site(rep):
    from scipy import stats
    out = {}
    for col in ["flag_outlier", "flag_outlier_global", "flag_outlier_site"]:
        for by in ["SITE_ID", "label", "qc_decision"]:
            ct = pd.crosstab(rep[by], rep[col])
            if ct.shape[1] < 2 or ct.shape[0] < 2:
                out[(col, by)] = {"chi2": np.nan, "p": np.nan, "catatan": "tidak dapat diuji (satu kategori)"}
                continue
            chi2, p, dof, exp = stats.chi2_contingency(ct)
            out[(col, by)] = {"chi2": chi2, "p": p, "dof": dof, "%expected<5": float((exp < 5).mean() * 100)}
            print(f"\n{col} × {by}"); display(ct)
    res = pd.DataFrame(out).T
    display(res)
    subsets = {"pass_only": rep[rep["pass_only"]], "pass_plus_review": rep}
    display(pd.DataFrame({k: {"n": len(v), "n_ASD": int(v["label"].sum()), "n_TD": int((v["label"] == 0).sum()),
                              "n_situs": v["SITE_ID"].nunique(), "outlier": int(v["flag_outlier"].sum())}
                          for k, v in subsets.items()}).T)
    return res


if RUN_QC_ANALYSIS:
    QC_ANALYSIS = qc_vs_label_site(pd.read_csv(QC_REPORT_PATH))
else:
    print("RUN_QC_ANALYSIS = False: analisis dijalankan setelah batch penuh.")

### CELL 13 — `preprocess_config.json` dan verifikasi akhir

**Tujuan:** menulis parameter, versi tool, seed, tanggal, box crop, dan ringkasan pilot ke `preprocess_config.json`; memastikan hash master dan `folds.csv` tidak berubah.

**Output:** `preprocess_config.json`.

In [ ]:
config = {
    "created": datetime.datetime.now().isoformat(timespec="seconds"), "seed": SEED,
    "versions": {k: v for k, v in ENV.items() if k != "hdbet_venv"}, "hdbet_venv": VENV_ENV,
    "pipeline": {"target_axcodes": TARGET_AXCODES, "hdbet_input_axcodes": HDBET_INPUT_AXCODES,
                 "hdbet_device": HDBET_DEVICE, "hdbet_tta": HDBET_TTA, "hdbet_chunk": HDBET_CHUNK, "hdbet_mode": HDBET_MODE,
                 "clip_negative_before_n4": CLIP_NEGATIVE_BEFORE_N4, "n4_params": N4_PARAMS,
                 "registration": REG_TRANSFORM, "ants_random_seed": ANTS_RANDOM_SEED, "itk_threads": ANTS_THREADS,
                 "template": f"nilearn {ENV['nilearn']} load_mni152_template (MNI152 2009, nilearn)",
                 "template_res_reg_mm": TEMPLATE_RES_REG_MM, "target_res_mm": TARGET_RES_MM,
                 "mask_threshold": MASK_THRESHOLD, "clip_percentiles": CLIP_PERCENTILES, "intensity": "z-score di dalam mask, latar 0",
                 "target_shape": [1, *TARGET_SHAPE], "dtype": "float32",
                 "crop": [[s.start, s.stop] for s in CROP], "pad": PAD, "crop_margin_vox": CROP_MARGIN_VOX},
    "qc": {"outlier_z": OUTLIER_Z, "min_n_global_z": MIN_N_GLOBAL_Z, "min_n_site_z": MIN_N_SITE_Z, "identical_atol": IDENTICAL_ATOL},
    "pilot": {"ids": PILOT_IDS, "n_ok": int(len(ok)), "n_failed": int(len(failed_pilot)),
              "mean_time_per_subject_s": mean_total, "wall_s": PILOT_WALL, "hdbet_runs": VRAM_LOG, "ram_log": RAM_LOG,
              "estimated_total_hours": mean_total * len(subjects) / 3600},
    "test_subject": {"SUB_ID": test_sid, "timing_s": timing, "vram_peak_mib": test_rec.get("vram_peak_mib")},
    "hdbet_runner_validation": VALIDATION,
    "hdbet_runner": {"mode": HDBET_MODE, "np_preprocessing": HDBET_NP_PREPROCESSING, "np_export": HDBET_NP_EXPORT,
                     "min_free_ram_gb": MIN_FREE_RAM_GB, "script": "src/hdbet_runner.py"},
    "perlu_dicek": ["HDBET_INPUT_AXCODES=LAS sebagai padanan fslreorient2std",
                    "N4 tanpa mask (default antspyx)", "determinisme ANTs multi-thread", "IDENTICAL_ATOL"],
}
CONFIG_JSON_PATH.write_text(json.dumps(config, indent=1, default=str), encoding="utf-8")

HASH_MASTER_AFTER, HASH_FOLDS_AFTER = sha256_file(CLEANED_PATH), sha256_file(FOLDS_PATH)
print("SHA-256 master awal/akhir:", HASH_MASTER_BEFORE, "/", HASH_MASTER_AFTER)
print("SHA-256 folds  awal/akhir:", HASH_FOLDS_BEFORE, "/", HASH_FOLDS_AFTER)
assert HASH_MASTER_BEFORE == HASH_MASTER_AFTER and HASH_FOLDS_BEFORE == HASH_FOLDS_AFTER

### CELL 13b — Verifikasi akhir

In [ ]:
assert json.loads(CONFIG_JSON_PATH.read_text(encoding="utf-8"))["seed"] == SEED
for p in (HEADER_CHECK_PATH, QC_REPORT_PATH, CONFIG_JSON_PATH):
    assert p.exists(), p
n_failed_total = len(pd.read_csv(FAILED_PATH)) if FAILED_PATH.exists() else 0
if not FAILED_PATH.exists():
    pd.DataFrame(columns=FAILED_COLS).to_csv(FAILED_PATH, index=False)
print("=== RINGKASAN 05c (sampai pilot) ===")
print(f"Subjek valid: {sum(output_valid(int(s)) for s in subjects['SUB_ID'])} / {len(subjects)} | gagal: {n_failed_total}")
print(f"Test {test_sid}: {round(test_rec['time_total'], 1)} s | VRAM puncak {test_rec.get('vram_peak_mib'):.0f} MiB")
print(f"Pilot: ok {len(ok)} / {len(PILOT_IDS)} | estimasi total {mean_total * len(subjects) / 3600:.1f} jam")
print("Hash master & folds: tidak berubah")
print("Output:", [p.relative_to(PROJECT_ROOT).as_posix() for p in (VOL_DIR, MASK_DIR, XFM_DIR, META_DIR, FIG_DIR,
                                                                    QC_REPORT_PATH, FAILED_PATH, HEADER_CHECK_PATH, CONFIG_JSON_PATH)])